In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('insurance.csv')
print(df)

      age     sex     bmi  children smoker     region      charges
0      19  female  27.900         0    yes  southwest  16884.92400
1      18    male  33.770         1     no  southeast   1725.55230
2      28    male  33.000         3     no  southeast   4449.46200
3      33    male  22.705         0     no  northwest  21984.47061
4      32    male  28.880         0     no  northwest   3866.85520
...   ...     ...     ...       ...    ...        ...          ...
1333   50    male  30.970         3     no  northwest  10600.54830
1334   18  female  31.920         0     no  northeast   2205.98080
1335   18  female  36.850         0     no  southeast   1629.83350
1336   21  female  25.800         0     no  southwest   2007.94500
1337   61  female  29.070         0    yes  northwest  29141.36030

[1338 rows x 7 columns]


In [3]:
df.drop_duplicates(inplace = True)

In [4]:
df.shape

(1337, 7)

In [5]:
df['sex'].value_counts()

sex
male      675
female    662
Name: count, dtype: int64

In [6]:
df['sex'] = df['sex'].map({"male" : 0, "female" : 1})

In [7]:
df['sex'].head()

0    1
1    0
2    0
3    0
4    0
Name: sex, dtype: int64

In [8]:
df['smoker'] = df['smoker'].map({"yes" : 0, "no" : 1})

In [9]:
df['smoker'].head()

0    0
1    1
2    1
3    1
4    1
Name: smoker, dtype: int64

In [10]:
df.rename(
    columns = {
        'sex' : 'is_female',
        'smoker' : 'is_smoker'
    },
    inplace = True
)

In [11]:
df['region'].value_counts()

region
southeast    364
southwest    325
northwest    324
northeast    324
Name: count, dtype: int64

In [22]:
df = pd.get_dummies(df, columns = ['region'])

In [16]:
df['bmi_category'] = pd.cut(
    df['bmi'],
    bins = [0, 18.5, 24.9, 29.9, float('inf')],
    labels = ['underweight', 'normal', 'overweight', 'obese']
)

In [17]:
df = pd.get_dummies(df, columns = ['bmi_category'])

In [23]:
df.head()

,age,is_female,bmi,children,is_smoker,charges,region_northeast,region_northwest,region_southeast,region_southwest
0,-1.440418,1,-0.453160,-0.909234,0,16884.92400,False,False,False,True
1,-1.511647,0,0.509422,-0.079442,1,1725.55230,False,False,True,False
2,-0.799350,0,0.383155,1.580143,1,4449.46200,False,False,True,False
3,-0.443201,0,-1.305052,-0.909234,1,21984.47061,False,True,False,False
4,-0.514431,0,-0.292456,-0.909234,1,3866.85520,False,True,False,False


In [24]:
df = df.astype(int)

In [25]:
df.shape

(1337, 10)

In [19]:
from sklearn.preprocessing import StandardScaler


In [20]:
cols = ['age', 'bmi', 'children']
scaler = StandardScaler()
df[cols] = scaler.fit_transform(df[cols])

In [26]:
print(df)

      age  is_female  bmi  children  is_smoker  charges  region_northeast  \
0      -1          1    0         0          0    16884                 0   
1      -1          0    0         0          1     1725                 0   
2       0          0    0         1          1     4449                 0   
3       0          0   -1         0          1    21984                 0   
4       0          0    0         0          1     3866                 0   
...   ...        ...  ...       ...        ...      ...               ...   
1333    0          0    0         1          1    10600                 0   
1334   -1          1    0         0          1     2205                 1   
1335   -1          1    1         0          1     1629                 0   
1336   -1          1    0         0          1     2007                 0   
1337    1          1    0         0          0    29141                 0   

      region_northwest  region_southeast  region_southwest  
0             

In [61]:
from scipy.stats import pearsonr

selected_features = ['age', 'is_female', 'bmi', 'children', 'is_smoker',
       'region_northeast', 'region_northwest', 'region_southeast',
       'region_southwest', 'bmi_category_underweight', 'bmi_category_normal',
       'bmi_category_overweight', 'bmi_category_obese'
]

correlations = {
    feature: pearsonr(df[feature], df['charges'])[0]
    for feature in selected_features
}

correlation_df = pd.DataFrame(list(correlations.items()), columns = ['Feature', 'Pearson Correlation'])
correlation_df.sort_values(by = 'Pearson Correlation', ascending = False)
print(correlation_df)

                     Feature  Pearson Correlation
0                        age             0.298309
1                  is_female            -0.058046
2                        bmi             0.196236
3                   children             0.067390
4                  is_smoker            -0.787234
5           region_northeast             0.005946
6           region_northwest            -0.038695
7           region_southeast             0.073577
8           region_southwest            -0.043637
9   bmi_category_underweight            -0.048225
10       bmi_category_normal            -0.105656
11   bmi_category_overweight            -0.118280
12        bmi_category_obese             0.197660


In [62]:
categorical_features = ['is_female','is_smoker',
       'region_northeast', 'region_northwest', 'region_southeast',
       'region_southwest', 'bmi_category_underweight', 'bmi_category_normal',
       'bmi_category_overweight', 'bmi_category_obese'
]

In [63]:
from scipy.stats import chi2_contingency
alpha = 0.5
df['charges_bin'] = pd.qcut(df['charges'], q = 4, labels = False)
chi2_results = {}

for col in categorical_features :
    contingency = pd.crosstab(df[col], df['charges_bin'])
    chi2_stat , p_val, _, _ = chi2_contingency(contingency)
    decision = 'Reject Null (Keep Feature)' if p_val < alpha else 'Accept Null (Drop Feature)'
    chi2_results[col] = {
        'chi2_statistic': chi2_stat,
        'p_value': p_val,
        'Decision': decision
    }

chi2_df = pd.DataFrame(chi2_results).T
chi2_df = chi2_df.sort_values(by = 'p_value')
print(chi2_df)

                         chi2_statistic   p_value                    Decision
is_smoker                    848.219178       0.0  Reject Null (Keep Feature)
region_southeast              15.998167  0.001135  Reject Null (Keep Feature)
is_female                     10.258784   0.01649  Reject Null (Keep Feature)
bmi_category_obese             7.654464   0.05372  Reject Null (Keep Feature)
region_northeast               6.438442  0.092122  Reject Null (Keep Feature)
region_southwest               5.091893  0.165191  Reject Null (Keep Feature)
bmi_category_underweight       4.384749  0.222804  Reject Null (Keep Feature)
bmi_category_normal            4.263673  0.234364  Reject Null (Keep Feature)
bmi_category_overweight        4.201575  0.240504  Reject Null (Keep Feature)
region_northwest                1.13424  0.768815  Accept Null (Drop Feature)


In [99]:
final_df = df[['age', 'is_female', 'bmi', 'children', 'is_smoker',
       'region_northeast', 'region_southeast',
       'region_southwest', 'bmi_category_underweight', 'bmi_category_normal',
       'bmi_category_overweight', 'bmi_category_obese'
]]

KeyError: "['is_female', 'is_smoker', 'region_northeast', 'region_southeast', 'region_southwest', 'bmi_category_underweight', 'bmi_category_normal', 'bmi_category_overweight', 'bmi_category_obese'] not in index"

In [26]:
final_df

,age,is_female,bmi,children,is_smoker,region_northeast,region_southeast,region_southwest,bmi_category_underweight,bmi_category_normal,bmi_category_overweight,bmi_category_obese
0,-1.440418,1,-0.517949,-0.909234,0,0,0,1,0,0,1,0
1,-1.511647,0,0.462463,-0.079442,1,0,1,0,0,0,0,1
2,-0.799350,0,0.462463,1.580143,1,0,1,0,0,0,0,1
3,-0.443201,0,-1.334960,-0.909234,1,0,0,0,0,1,0,0
4,-0.514431,0,-0.354547,-0.909234,1,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
1333,0.767704,0,-0.027743,1.580143,1,0,0,0,0,0,0,1
1334,-1.511647,1,0.135659,-0.909234,1,1,0,0,0,0,0,1
1335,-1.511647,1,0.952670,-0.909234,1,0,1,0,0,0,0,1
1336,-1.297958,1,-0.844753,-0.909234,1,0,0,1,0,0,1,0


In [14]:
from sklearn.model_selection import train_test_split

In [42]:
X = df.drop(columns = ['is_smoker'], axis = 1)
y = df['is_smoker']

In [43]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.33, random_state=42)

In [64]:
from sklearn import svm

In [72]:
model = svm.SVC()
model.fit(X_train, y_train) 

SVC()

In [73]:
model_predicted = model.predict(X_test)

In [31]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [52]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, model_predicted)
rmse = np.sqrt(mean_squared_error(y_test, model_predicted))
r2 = r2_score(y_test, model_predicted)

print(mae, rmse, r2)

0.058823529411764705 0.24253562503633297 0.6310991268618388


In [53]:
n = len(y_test)      # number of samples
p = X.shape[1]       # number of predictors

adjusted_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print(adjusted_r2)

0.6234136920047937


In [74]:
accuracy_score(y_test, model_predicted)

0.9276018099547512

In [75]:
confusion_matrix(y_test, model_predicted)

array([[ 81,   7],
       [ 25, 329]])

In [76]:
print(classification_report(y_test, model_predicted))

              precision    recall  f1-score   support

           0       0.76      0.92      0.84        88
           1       0.98      0.93      0.95       354

    accuracy                           0.93       442
   macro avg       0.87      0.92      0.89       442
weighted avg       0.94      0.93      0.93       442

